In [0]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
repo_root = next(
    (
        path for path in (current, *current.parents)
        if (path / "transformations" / "silver").is_dir()
        and (path / "sources" / "reference").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("No se encontró la raíz del repositorio.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from transformations.silver.sqlserver_transform import (
    read_sqlserver_tables,
    validate_sqlserver_tables,
    add_sqlserver_provenance,
    write_sqlserver_delta,
)

print("PASS: funciones de SQL Server importadas.")

In [0]:
ingestion_run_id = "89d277f4-ddee-49a2-9d9f-588efacf9788"

bronze_root = (
    "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
    "sqlserver/techretail_oltp/initial/"
    f"run_id={ingestion_run_id}"
)

frames = read_sqlserver_tables(spark, bronze_root)
checks = validate_sqlserver_tables(frames)

print(f"PASS: {len(checks)} controles superados.")

for entity, frame in frames.items():
    print(f"{entity}: {frame.count()} filas")

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

processing_run_id = str(uuid4())
processed_at = datetime.now(timezone.utc)

silver_root = (
    "abfss://silver@starlpdevdiego01.dfs.core.windows.net"
)

transformed = add_sqlserver_provenance(
    frames,
    bronze_root,
    ingestion_run_id,
    processing_run_id,
    processed_at,
)

results = write_sqlserver_delta(
    spark,
    transformed,
    silver_root,
)

for result in results:
    print(
        f"PASS: {result['entity']}, "
        f"{result['row_count']} filas verificadas."
    )

print("Processing run ID:", processing_run_id)

In [0]:
silver_products = spark.read.format("delta").load(
    f"{silver_root}/techretail/products"
)

display(
    silver_products.select(
        "product_id",
        "product_name",
        "list_price",
        "source_system",
        "processing_run_id",
        "processed_at",
    ).orderBy("product_id").limit(10)
)

In [0]:
spark.sql("""
    SELECT
        current_catalog() AS catalog,
        current_schema() AS schema
""").show(truncate=False)

spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
catalog = "dbw_arlp_dev_diego01"
schema = "silver_techretail"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

silver_root = (
    "abfss://silver@starlpdevdiego01.dfs.core.windows.net"
)

for entity in (
    "categories", "products", "customers", "orders", "order_items"
):
    target = f"{silver_root}/techretail/{entity}"

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{entity}
        USING DELTA
        LOCATION '{target}'
    """)

    print(f"Registrada: {catalog}.{schema}.{entity}")

In [0]:
spark.sql("""
    SELECT product_id, product_name, list_price
    FROM dbw_arlp_dev_diego01.silver_techretail.products
    ORDER BY product_id
    LIMIT 10
""").show(truncate=False)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
        "dummyjson/"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
        "dummyjson/products/"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
        "dummyjson/products/ingestion_date=2026-10-11/"
    )
)